In [141]:
%pip install xgboost

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, log_loss




np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.grid'] = True
sns.set_theme(style='whitegrid')
print('Libraries loaded ✓')

/Users/ksilamahdi/.zshenv:export:2: not valid in this context: Support/JetBrains/Toolbox/scripts:/Users/ksilamahdi/mongodb-macos-aarch64-8.0.3/bin
Note: you may need to restart the kernel to use updated packages.
Libraries loaded ✓


In [142]:
# Load datasets
results_df = pd.read_csv("results.csv")
elo_df = pd.read_csv("eloratings.csv")

# Display basic info
print("Results shape:", results_df.shape)
print("ELO shape:", elo_df.shape)

# Preview
results_df.head()

Results shape: (49215, 9)
ELO shape: (6678, 4)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False


In [143]:
# Check columns
print(results_df.columns)
print(elo_df.columns)

# Check missing values
print("\nMissing values (results):")
print(results_df.isnull().sum())

print("\nMissing values (elo):")
print(elo_df.isnull().sum())

Index(['date', 'home_team', 'away_team', 'home_score', 'away_score',
       'tournament', 'city', 'country', 'neutral'],
      dtype='object')
Index(['date', 'team', 'rating', 'change'], dtype='object')

Missing values (results):
date          0
home_team     0
away_team     0
home_score    0
away_score    0
tournament    0
city          0
country       0
neutral       0
dtype: int64

Missing values (elo):
date       0
team       0
rating    31
change     0
dtype: int64


In [144]:
results_df['date'] = pd.to_datetime(results_df['date'])
elo_df['date'] = pd.to_datetime(elo_df['date'])

In [145]:
results_df = results_df[[
    'date', 'home_team', 'away_team',
    'home_score', 'away_score', 'tournament', 'neutral'
]]

elo_df = elo_df[['date', 'team', 'rating']]

In [146]:
# 1 = home win, 0 = not win
results_df['home_win'] = (results_df['home_score'] > results_df['away_score']).astype(int)

In [147]:
# Get latest ELO per team
latest_elo = elo_df.sort_values('date').groupby('team').tail(1)

# Rename for merging
latest_elo = latest_elo[['team', 'rating']]
latest_elo.rename(columns={'rating': 'elo'}, inplace=True)

In [148]:
# Merge home team ELO
results_df = results_df.merge(
    latest_elo,
    left_on='home_team',
    right_on='team',
    how='left'
).rename(columns={'elo': 'home_elo'}).drop(columns=['team'])

# Merge away team ELO
results_df = results_df.merge(
    latest_elo,
    left_on='away_team',
    right_on='team',
    how='left'
).rename(columns={'elo': 'away_elo'}).drop(columns=['team'])
 

In [149]:
results_df['elo_diff'] = results_df['home_elo'] - results_df['away_elo']

In [150]:
results_df = results_df.dropna()
results_df.head()

,date,home_team,away_team,home_score,away_score,tournament,neutral,home_win,home_elo,away_elo,elo_diff
0,1872-11-30,Scotland,England,0,0,Friendly,False,0,1790.0,2042.0,-252.0
1,1873-03-08,England,Scotland,4,2,Friendly,False,1,2042.0,1790.0,252.0
2,1874-03-07,Scotland,England,2,1,Friendly,False,1,1790.0,2042.0,-252.0
3,1875-03-06,England,Scotland,2,2,Friendly,False,0,2042.0,1790.0,252.0
4,1876-03-04,Scotland,England,3,0,Friendly,False,1,1790.0,2042.0,-252.0


In [151]:
# Features
X = results_df[['home_elo', 'away_elo', 'elo_diff', 'neutral']]

# Target
y = results_df['home_win']

In [152]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (26296, 4)
Test size: (6574, 4)


In [153]:
# Initialize XGBoost model
model = XGBClassifier(
    objective='multi:softprob',  # for multi-class probability
    num_class=3,                 # win/draw/loss
    eval_metric='mlogloss',
    use_label_encoder=False,
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)


In [154]:
# Train the model
model.fit(X_train, y_train)

# Predictions (probabilities)
y_pred_prob = model.predict_proba(X_test)

# Predicted class
y_pred_class = np.argmax(y_pred_prob, axis=1)

# Evaluate
acc = accuracy_score(y_test, y_pred_class)
loss = log_loss(y_test, y_pred_prob, labels=[0, 1, 2])
print(f"Accuracy: {acc:.3f}")
print(f"Log Loss: {loss:.3f}")

Accuracy: 0.680
Log Loss: 0.597


In [155]:
def predict_match(home_team, away_team, neutral=True):
    home_elo_val = latest_elo.loc[latest_elo['team'] == home_team, 'elo']
    away_elo_val = latest_elo.loc[latest_elo['team'] == away_team, 'elo']

    if home_elo_val.empty or away_elo_val.empty:
        missing = [t for t, v in [(home_team, home_elo_val), (away_team, away_elo_val)] if v.empty]
        raise ValueError(f"Cannot find ELO for team(s): {', '.join(missing)}")

    home_elo_val = home_elo_val.iloc[0]
    away_elo_val = away_elo_val.iloc[0]
    elo_diff_val = home_elo_val - away_elo_val

    match_data = pd.DataFrame([{
        'home_elo': home_elo_val,
        'away_elo': away_elo_val,
        'elo_diff': elo_diff_val,
        'neutral': neutral
    }])

    prob = model.predict_proba(match_data)[0][1]
    
    print(f"{home_team} vs {away_team}")
    print(f"Win probability for {home_team}: {prob:.2f}")
    
    if prob >= 0.5:
        print(f"Predicted winner: {home_team}")
    else:
        print(f"Predicted winner: {away_team}")



In [156]:
predict_match("France", "Brazil")
predict_match("Argentina", "England")
predict_match("Germany", "Spain")

France vs Brazil
Win probability for France: 0.34
Predicted winner: Brazil
Argentina vs England
Win probability for Argentina: 0.46
Predicted winner: England
Germany vs Spain
Win probability for Germany: 0.12
Predicted winner: Spain


In [157]:
numeric_features = ['home_elo', 'away_elo', 'elo_diff']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features)
    ],
    remainder='passthrough'  # keeps 'neutral'
)

In [158]:
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression())
])

In [159]:
model.fit(X_train, y_train)
print("Model trained ✓")

Model trained ✓


In [160]:
y_pred = model.predict(X_test)

In [161]:
y_pred_binary = (y_pred >= 0.5).astype(int)

In [162]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("MSE:", mse)
print("R2 Score:", r2)

MAE: 0.3439306358381503
MSE: 0.3439306358381503
R2 Score: -0.377651623283938


In [163]:
def predict_match(team1, team2):
    # Get ELOs
    elo1 = latest_elo[latest_elo['team'] == team1]['elo'].values[0]
    elo2 = latest_elo[latest_elo['team'] == team2]['elo'].values[0]
    
    # Create input
    match_data = pd.DataFrame([{
        'home_elo': elo1,
        'away_elo': elo2,
        'elo_diff': elo1 - elo2,
        'neutral': True  # World Cup is neutral
    }])
    
    # Predict
    pred = model.predict(match_data)[0]
    
    # Convert to probability-like output
    prob = model.predict_proba(match_data)[0][1]
    
    print(f"{team1} vs {team2}")
    print(f"Win probability for {team1}: {prob:.2f}")
    
    if prob >= 0.5:
        print(f"Predicted winner: {team1}")
    else:
        print(f"Predicted winner: {team2}")

In [164]:
predict_match("France", "Brazil")
predict_match("Argentina", "England")
predict_match("Germany", "Spain")

France vs Brazil
Win probability for France: 0.45
Predicted winner: Brazil
Argentina vs England
Win probability for Argentina: 0.44
Predicted winner: England
Germany vs Spain
Win probability for Germany: 0.24
Predicted winner: Spain
